# Learning Algorithms

Notebook này trình bày quy trình thực hành trực tiếp cho nhóm **Learning Algorithms** trên bài toán **Grid-world Navigation 8x8**. Khác với Planning, các thuật toán Learning không giả định agent biết đầy đủ transition model của MDP. Agent phải học từ kinh nghiệm tương tác với môi trường.

Mỗi lần tương tác tạo ra một sampled transition:

$$
(S_t, A_t, R_{t+1}, S_{t+1})
$$

Trong training, Learning agent tương tác với environment thông qua:

```python
env.reset()
env.step(action)
```

và **không dùng trực tiếp** `get_transitions(state, action)`. Đây là khác biệt cốt lõi so với Planning Algorithms, nơi agent có thể truy cập đầy đủ $P(s' \mid s,a)$ và $r(s,a,s')$.

Notebook này tập trung vào bốn thuật toán Learning chính:

1. **TD(0)**: prediction method học $V^\pi(s)$ từ one-step TD updates.
2. **TD($\lambda$)**: prediction method mở rộng TD(0) bằng eligibility traces.
3. **SARSA**: on-policy control method học $Q(s,a)$ theo chính policy đang tương tác.
4. **Q-learning**: off-policy control method học $Q(s,a)$ với greedy target.

Về baseline đánh giá, notebook dùng đúng cặp so sánh:

- **PolicyEvaluation** từ Planning làm baseline cho TD(0) và TD($\lambda$), vì cả ba đều là prediction/evaluation methods dưới một policy.
- **ValueIteration** từ Planning làm baseline cho SARSA và Q-learning, vì các thuật toán này là control methods hướng tới policy tốt hoặc tối ưu.

Luồng chính của notebook là: tạo environment → khởi tạo model → train model → lấy metrics → đánh giá với baseline phù hợp → gọi trực tiếp visualization helpers. Logs JSON chỉ là optional/reference, không phải luồng chính.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import json
from pathlib import Path
from time import perf_counter

from envs.learning_grid_world import LearningGridWorld
from envs.planning_grid_world import PlanningGridWorld
from agents.planning import PolicyEvaluation, ValueIteration
from agents.learning import TDZero, TDLambda, SARSA, QLearning
from utils.metrics import mean_squared_error, policy_agreement
from utils.visualization import (
    plot_grid_world_layout,
    plot_value_heatmap,
    plot_policy_arrows,
    plot_learning_curve,
    plot_moving_average,
    plot_episode_steps,
    plot_td_error_curve,
    plot_td_mse_curve,
    plot_success_trap_curves,
    plot_comparison_bar,
)

NOTEBOOK_FIGURE_DIR = PROJECT_ROOT / "report" / "figures" / "learning"
NOTEBOOK_VERBOSE = 1
NOTEBOOK_LOG_INTERVAL = 100
NOTEBOOK_WINDOW_SIZE = 100
LEARNING_EPISODES = 1000
MAX_STEPS_PER_EPISODE = 256
RANDOM_SEED = 42

NOTEBOOK_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURE_DIR


In [ ]:
def show_metrics(metrics, keys=None):
    if metrics is None:
        print("No metrics available.")
        return
    selected = metrics if keys is None else {key: metrics.get(key) for key in keys}
    for key, value in selected.items():
        print(f"{key}: {value}")


def load_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


## Thiết kế Grid-world 8x8

Trong project này, môi trường Learning cũng là **Grid-world Navigation 8x8**. Đây là môi trường rời rạc, trong đó mỗi state là một tọa độ:

$$
s = (row, col)
$$

Agent có thể chọn một trong bốn action:

$$
\mathcal{A} = \{up, down, left, right\}
$$

Bố cục môi trường gồm:

- **Start state**: vị trí bắt đầu episode.
- **Goal states**: trạng thái kết thúc tốt, nhận reward dương.
- **Trap states**: trạng thái kết thúc xấu, nhận reward âm.
- **Wall states**: ô chướng ngại, agent không thể đi xuyên qua.
- **Step reward**: reward mỗi bước, khuyến khích agent tìm đường đi hiệu quả.

Về mặt hình thức, Grid-world vẫn là một MDP:

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, r, \gamma)
$$

Tuy nhiên, khác với Planning, các thuật toán Learning **không dùng** `get_transitions()` trong training. Agent chỉ quan sát sampled transitions thông qua `reset()` và `step(action)`. Vì vậy, các đường cong return, success rate, trap rate và learned policy ở các phần sau phản ánh quá trình học từ sample, không phải nghiệm tính trực tiếp từ model.


In [ ]:
learning_env = LearningGridWorld(seed=RANDOM_SEED)
learning_env

In [ ]:
print("Grid size:", learning_env.grid_size)
print("Start state:", learning_env.start_state)
print("Goal states:", learning_env.goal_states)
print("Trap states:", learning_env.trap_states)
print("Wall states:", learning_env.wall_states)
print("Actions:", learning_env.actions)
print("Reward config:", learning_env.reward_config)
print("Gamma:", learning_env.gamma)
print("Number of states:", learning_env.num_states())

In [ ]:
layout_path = NOTEBOOK_FIGURE_DIR / "grid_world_layout.png"
plot_grid_world_layout(
    learning_env,
    title="Grid-world 8x8 Layout",
    save_path=layout_path,
    show=True,
)


Bố cục trên cho thấy vị trí **Start**, **Goal**, **Trap** và **Wall**. Cách bố trí này sẽ ảnh hưởng trực tiếp đến return curves, success/trap rates và learned policy của các thuật toán control. Agent cần học cách đi tới Goal, tránh Trap và đi vòng qua Wall chỉ từ sampled interaction.

## Chuẩn bị baseline đánh giá Learning Algorithms

Learning notebook dùng baseline Planning **chỉ để đánh giá sau training**, không dùng trong quá trình học của model-free algorithms.

- TD(0) và TD($\lambda$) là prediction methods, nên baseline đúng là $V^\pi$ từ `PolicyEvaluation`.
- SARSA và Q-learning là control methods, nên baseline đúng là $V^*$ và $\pi^*$ từ `ValueIteration`.
- Trong training, các Learning algorithms vẫn chỉ dùng `reset()` và `step(action)`.

Vì `TDZero` và `TDLambda` dùng uniform random policy khi không truyền policy, baseline `PolicyEvaluation(policy=None)` cũng dùng uniform random policy để so sánh đúng bài toán prediction.


In [ ]:
baseline_env = PlanningGridWorld(seed=RANDOM_SEED)

baseline_policy_eval = PolicyEvaluation(
    env=baseline_env,
    policy=None,  # uniform random policy mặc định, cùng tinh thần với TD policy mặc định
    theta=1e-6,
    max_iterations=1000,
    verbose=0,
)
baseline_policy_eval.run()
baseline_v_pi = baseline_policy_eval.get_value_function()

baseline_vi = ValueIteration(
    env=baseline_env,
    theta=1e-6,
    max_iterations=1000,
    verbose=0,
)
baseline_vi.run()
baseline_v_star = baseline_vi.get_value_function()
baseline_pi_star = baseline_vi.get_policy()

print("Policy Evaluation baseline status:", baseline_policy_eval.get_metrics().get("status"))
print("Value Iteration baseline status:", baseline_vi.get_metrics().get("status"))

## Xây dựng và train các thuật toán Learning

Trong phần này, ta lần lượt xây dựng và train bốn thuật toán Learning:

1. TD(0)
2. TD($\lambda$)
3. SARSA
4. Q-learning

Với mỗi thuật toán, notebook đi theo quy trình: nhắc lại mục tiêu và công thức, khởi tạo model từ `agents.learning`, gọi `.train()`, lấy value/Q/policy/metrics, đánh giá bằng baseline phù hợp và gọi trực tiếp visualization helpers.

### TD(0)

**TD(0)** là prediction algorithm dùng để học $V^\pi(s)$ từ sampled experience. Thuật toán không biết transition model $P(s' \mid s,a)$ và không gọi `get_transitions()` trong training.

TD error:

$$
\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)
$$

Update rule:

$$
V(S_t) \leftarrow V(S_t) + \alpha \delta_t
$$

Nếu $S_{t+1}$ là terminal state thì target không bootstrap từ $V(S_{t+1})$. Baseline đánh giá là $V^\pi$ từ `PolicyEvaluation`. TD(0) không tìm policy tối ưu; nó chỉ đánh giá một behavior policy từ sample.


In [ ]:
td_zero_env = LearningGridWorld(seed=RANDOM_SEED)

td_zero = TDZero(
    env=td_zero_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=td_zero_env.gamma,
    policy=None,
    baseline_value_function=baseline_v_pi,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

td_zero_result = td_zero.train()
td_zero_values = td_zero.get_value_function()
td_zero_metrics = td_zero.get_metrics()

In [ ]:
# Đánh giá trực tiếp từ model vừa train.
td_zero_mse_direct = mean_squared_error(td_zero_values, baseline_v_pi)
td_zero_metrics["notebook_mse_vs_policy_evaluation"] = td_zero_mse_direct

show_metrics(td_zero_metrics, keys=[
    "episodes",
    "environment_steps",
    "final_mean_absolute_td_error",
    "mse_vs_policy_evaluation",
    "notebook_mse_vs_policy_evaluation",
    "runtime_sec",
])

In [ ]:
td0_mse_path = NOTEBOOK_FIGURE_DIR / "td_zero_mse_vs_pe.png"
td0_error_path = NOTEBOOK_FIGURE_DIR / "td_zero_td_error.png"
td0_value_path = NOTEBOOK_FIGURE_DIR / "td_zero_value_heatmap.png"

plot_td_mse_curve(
    td_zero_metrics["mse_checkpoint_episodes"],
    td_zero_metrics["mse_vs_policy_evaluation_checkpoints"],
    "TD(0) - MSE vs Policy Evaluation",
    td0_mse_path,
)
plot_td_error_curve(
    td_zero_metrics["mean_absolute_td_error_per_episode"],
    "TD(0) Mean Absolute TD Error",
    td0_error_path,
)
plot_value_heatmap(
    td_zero_values,
    td_zero_env,
    "TD(0) - V^pi Estimate",
    save_path=td0_value_path,
    show=True,
)


**Nhận xét sau khi chạy TD(0):**

- MSE vs PolicyEvaluation cho biết TD(0) học gần baseline $V^\pi$ đến đâu.
- TD error có thể dao động vì thuật toán học từ sampled trajectories; MSE checkpoints thường hữu ích hơn để đánh giá xu hướng prediction.
- Value heatmap biểu diễn ước lượng $V^\pi$ của policy đang được học, không phải value tối ưu.


### TD($\lambda$)

**TD($\lambda$)** mở rộng TD(0) bằng eligibility traces, cho phép TD error lan truyền ngược về nhiều state đã được thăm trước đó.

Eligibility trace:

$$
e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbf{1}\{S_t=s\}
$$

Update rule:

$$
V(s) \leftarrow V(s) + \alpha \delta_t e_t(s)
$$

Khi $\lambda=0$, TD($\lambda$) gần với TD(0). Khi $\lambda$ lớn hơn, reward signal có thể ảnh hưởng đến nhiều state trước đó hơn. Tuy nhiên TD($\lambda$) không đảm bảo luôn tốt hơn TD(0); kết quả phụ thuộc $\lambda$, $\alpha$, trajectory và môi trường. Baseline đánh giá vẫn là `PolicyEvaluation`.


In [ ]:
td_lambda_env = LearningGridWorld(seed=RANDOM_SEED)

td_lambda = TDLambda(
    env=td_lambda_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=td_lambda_env.gamma,
    lambda_=0.7,
    policy=None,
    baseline_value_function=baseline_v_pi,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

td_lambda_result = td_lambda.train()
td_lambda_values = td_lambda.get_value_function()
td_lambda_metrics = td_lambda.get_metrics()

In [ ]:
td_lambda_mse_direct = mean_squared_error(td_lambda_values, baseline_v_pi)
td_lambda_metrics["notebook_mse_vs_policy_evaluation"] = td_lambda_mse_direct

show_metrics(td_lambda_metrics, keys=[
    "episodes",
    "lambda",
    "environment_steps",
    "final_mean_absolute_td_error",
    "mse_vs_policy_evaluation",
    "notebook_mse_vs_policy_evaluation",
    "runtime_sec",
])

In [ ]:
tdl_mse_path = NOTEBOOK_FIGURE_DIR / "td_lambda_mse_vs_pe.png"
tdl_error_path = NOTEBOOK_FIGURE_DIR / "td_lambda_td_error.png"
tdl_value_path = NOTEBOOK_FIGURE_DIR / "td_lambda_value_heatmap.png"

plot_td_mse_curve(
    td_lambda_metrics["mse_checkpoint_episodes"],
    td_lambda_metrics["mse_vs_policy_evaluation_checkpoints"],
    "TD(lambda) - MSE vs Policy Evaluation",
    tdl_mse_path,
)
plot_td_error_curve(
    td_lambda_metrics["mean_absolute_td_error_per_episode"],
    "TD(lambda) Mean Absolute TD Error",
    tdl_error_path,
)
plot_value_heatmap(
    td_lambda_values,
    td_lambda_env,
    "TD(lambda) - V^pi Estimate",
    save_path=tdl_value_path,
    show=True,
)


**Nhận xét sau khi chạy TD($\lambda$):**

- So sánh MSE vs PolicyEvaluation với TD(0) để xem eligibility traces có giúp prediction gần baseline hơn không.
- Với `lambda_=0.7`, kết quả cần được hiểu như một cấu hình cụ thể, không phải kết luận tổng quát rằng TD($\lambda$) luôn tốt hơn TD(0).
- Nếu cần đánh giá ổn định hơn, dùng optional TD($\lambda$) sweep ở cuối notebook.


### SARSA

**SARSA** là on-policy control algorithm. Thuật toán học action-value function $Q(s,a)$ và policy từ trải nghiệm tương tác với environment.

Update rule:

$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)\right]
$$

Điểm quan trọng là $A_{t+1}$ là action thực sự được chọn bởi behavior/current policy. Vì vậy SARSA học giá trị theo chính policy đang dùng để tương tác, nên là **on-policy**. Baseline đánh giá là $V^*$ và $\pi^*$ từ `ValueIteration`.


In [ ]:
sarsa_env = LearningGridWorld(seed=RANDOM_SEED)

sarsa = SARSA(
    env=sarsa_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=sarsa_env.gamma,
    epsilon=0.1,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    window_size=NOTEBOOK_WINDOW_SIZE,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

sarsa_result = sarsa.train()
sarsa_q = sarsa.get_q_table()
sarsa_values = sarsa.get_value_function()
sarsa_policy = sarsa.get_policy()
sarsa_metrics = sarsa.get_metrics()

In [ ]:
non_terminal_states = [state for state in baseline_env.get_states() if not baseline_env.is_terminal(state)]

sarsa_mse_vs_vi = mean_squared_error(sarsa_values, baseline_v_star)
sarsa_agreement_vs_vi = policy_agreement(sarsa_policy, baseline_pi_star, non_terminal_states)
sarsa_metrics["notebook_mse_vs_value_iteration"] = sarsa_mse_vs_vi
sarsa_metrics["notebook_policy_agreement_vs_value_iteration"] = sarsa_agreement_vs_vi

show_metrics(sarsa_metrics, keys=[
    "episodes",
    "environment_steps",
    "training_avg_return",
    "final_window_avg_return",
    "final_window_success_rate",
    "final_window_trap_rate",
    "notebook_mse_vs_value_iteration",
    "notebook_policy_agreement_vs_value_iteration",
    "runtime_sec",
])

In [ ]:
sarsa_return_path = NOTEBOOK_FIGURE_DIR / "sarsa_episode_returns.png"
sarsa_moving_avg_path = NOTEBOOK_FIGURE_DIR / "sarsa_moving_average_returns.png"
sarsa_success_path = NOTEBOOK_FIGURE_DIR / "sarsa_success_trap.png"
sarsa_policy_path = NOTEBOOK_FIGURE_DIR / "sarsa_policy.png"
sarsa_value_path = NOTEBOOK_FIGURE_DIR / "sarsa_value_heatmap.png"

plot_learning_curve(sarsa_metrics["episode_returns"], "SARSA Episode Returns", sarsa_return_path)
plot_moving_average(sarsa_metrics["moving_average_returns"], "SARSA Moving Average Returns", sarsa_moving_avg_path)
plot_success_trap_curves(
    sarsa_metrics["window_metric_episodes"],
    sarsa_metrics["window_success_rates"],
    sarsa_metrics["window_trap_rates"],
    "SARSA Success/Trap Rates",
    sarsa_success_path,
)
plot_policy_arrows(
    sarsa_policy,
    sarsa_env,
    "SARSA Learned Policy",
    save_path=sarsa_policy_path,
    show=True,
)
plot_value_heatmap(
    sarsa_values,
    sarsa_env,
    "SARSA max_a Q(s,a)",
    save_path=sarsa_value_path,
    show=True,
)


**Nhận xét sau khi chạy SARSA:**

- Episode returns và moving average returns cho thấy policy có cải thiện qua training hay không.
- Success/trap curves giúp đánh giá behavior cuối training, không chỉ nhìn vào return trung bình.
- Vì SARSA là on-policy, learned policy phản ánh cả quá trình exploration trong training. Policy agreement với ValueIteration không nhất thiết phải bằng 1.0 để policy vẫn hữu ích trong rollout.


### Q-learning

**Q-learning** là off-policy control algorithm. Behavior policy có thể là epsilon-greedy để exploration, nhưng target update dùng greedy action ở next state.

Update rule:

$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma \max_{a'}Q(S_{t+1},a') - Q(S_t,A_t)\right]
$$

Khác với SARSA, Q-learning không dùng action thực sự $A_{t+1}$ trong target mà dùng $\max_{a'}Q(S_{t+1},a')$. Vì vậy nó học theo greedy target, nên là **off-policy**. Baseline đánh giá là `ValueIteration`.


In [ ]:
q_env = LearningGridWorld(seed=RANDOM_SEED)

q_learning = QLearning(
    env=q_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=q_env.gamma,
    epsilon=0.1,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    window_size=NOTEBOOK_WINDOW_SIZE,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

q_result = q_learning.train()
q_table = q_learning.get_q_table()
q_values = q_learning.get_value_function()
q_policy = q_learning.get_policy()
q_metrics = q_learning.get_metrics()

In [ ]:
q_mse_vs_vi = mean_squared_error(q_values, baseline_v_star)
q_agreement_vs_vi = policy_agreement(q_policy, baseline_pi_star, non_terminal_states)
q_metrics["notebook_mse_vs_value_iteration"] = q_mse_vs_vi
q_metrics["notebook_policy_agreement_vs_value_iteration"] = q_agreement_vs_vi

show_metrics(q_metrics, keys=[
    "episodes",
    "environment_steps",
    "training_avg_return",
    "final_window_avg_return",
    "final_window_success_rate",
    "final_window_trap_rate",
    "notebook_mse_vs_value_iteration",
    "notebook_policy_agreement_vs_value_iteration",
    "runtime_sec",
])

In [ ]:
q_return_path = NOTEBOOK_FIGURE_DIR / "q_learning_episode_returns.png"
q_moving_avg_path = NOTEBOOK_FIGURE_DIR / "q_learning_moving_average_returns.png"
q_success_path = NOTEBOOK_FIGURE_DIR / "q_learning_success_trap.png"
q_policy_path = NOTEBOOK_FIGURE_DIR / "q_learning_policy.png"
q_value_path = NOTEBOOK_FIGURE_DIR / "q_learning_value_heatmap.png"

plot_learning_curve(q_metrics["episode_returns"], "Q-learning Episode Returns", q_return_path)
plot_moving_average(q_metrics["moving_average_returns"], "Q-learning Moving Average Returns", q_moving_avg_path)
plot_success_trap_curves(
    q_metrics["window_metric_episodes"],
    q_metrics["window_success_rates"],
    q_metrics["window_trap_rates"],
    "Q-learning Success/Trap Rates",
    q_success_path,
)
plot_policy_arrows(
    q_policy,
    q_env,
    "Q-learning Learned Policy",
    save_path=q_policy_path,
    show=True,
)
plot_value_heatmap(
    q_values,
    q_env,
    "Q-learning max_a Q(s,a)",
    save_path=q_value_path,
    show=True,
)


**Nhận xét sau khi chạy Q-learning:**

- Q-learning học off-policy: behavior có exploration, nhưng target dùng greedy action.
- Return và success/trap curves cho thấy policy học được có cải thiện qua training hay không.
- Có thể đạt success rate cao dù policy agreement với ValueIteration chưa bằng 1.0, vì Grid-world có thể có nhiều action gần tối ưu hoặc nhiều đường đi tốt.
- Kết quả phụ thuộc episodes, alpha, epsilon và random seed.


## Đánh giá và so sánh Learning Algorithms

So sánh Learning phải giữ đúng bản chất thuật toán:

- TD(0) và TD($\lambda$) được so với `PolicyEvaluation`, vì đây là prediction/evaluation setting.
- SARSA và Q-learning được so với `ValueIteration`, vì đây là control setting.
- Không so TD methods trực tiếp với ValueIteration như thể chúng giải cùng bài toán.
- Không so SARSA/Q-learning trực tiếp với PolicyEvaluation như baseline optimal-control.

Các metrics dưới đây được lấy từ model vừa train trong notebook, không load logs cũ làm luồng chính.


In [ ]:
learning_comparison = {
    "TDZero_mse_vs_PE": td_zero_metrics.get("notebook_mse_vs_policy_evaluation"),
    "TDLambda_mse_vs_PE": td_lambda_metrics.get("notebook_mse_vs_policy_evaluation"),
    "SARSA_window_return": sarsa_metrics.get("final_window_avg_return"),
    "QLearning_window_return": q_metrics.get("final_window_avg_return"),
    "SARSA_window_success": sarsa_metrics.get("final_window_success_rate"),
    "QLearning_window_success": q_metrics.get("final_window_success_rate"),
    "SARSA_mse_vs_VI": sarsa_metrics.get("notebook_mse_vs_value_iteration"),
    "QLearning_mse_vs_VI": q_metrics.get("notebook_mse_vs_value_iteration"),
    "SARSA_policy_agreement": sarsa_metrics.get("notebook_policy_agreement_vs_value_iteration"),
    "QLearning_policy_agreement": q_metrics.get("notebook_policy_agreement_vs_value_iteration"),
}
show_metrics(learning_comparison)

In [ ]:
td_mse_comparison_path = NOTEBOOK_FIGURE_DIR / "td_mse_vs_policy_evaluation_comparison.png"
control_return_path = NOTEBOOK_FIGURE_DIR / "control_window_return_comparison.png"
control_success_path = NOTEBOOK_FIGURE_DIR / "control_window_success_comparison.png"
control_agreement_path = NOTEBOOK_FIGURE_DIR / "control_policy_agreement_comparison.png"
control_mse_path = NOTEBOOK_FIGURE_DIR / "control_mse_vs_value_iteration_comparison.png"

plot_comparison_bar(
    {
        "TD(0)": learning_comparison["TDZero_mse_vs_PE"],
        "TD(lambda)": learning_comparison["TDLambda_mse_vs_PE"],
    },
    "TD Prediction MSE vs Policy Evaluation",
    save_path=td_mse_comparison_path,
    ylabel="MSE",
    show=True,
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_window_return"],
        "Q-learning": learning_comparison["QLearning_window_return"],
    },
    "Control Final Window Return",
    save_path=control_return_path,
    ylabel="Return",
    show=True,
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_window_success"],
        "Q-learning": learning_comparison["QLearning_window_success"],
    },
    "Control Final Window Success Rate",
    save_path=control_success_path,
    ylabel="Success rate",
    show=True,
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_mse_vs_VI"],
        "Q-learning": learning_comparison["QLearning_mse_vs_VI"],
    },
    "Control MSE vs Value Iteration",
    save_path=control_mse_path,
    ylabel="MSE",
    show=True,
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_policy_agreement"],
        "Q-learning": learning_comparison["QLearning_policy_agreement"],
    },
    "Control Policy Agreement vs Value Iteration",
    save_path=control_agreement_path,
    ylabel="Agreement",
    show=True,
)


## Thực nghiệm mở rộng tùy chọn

Các phân tích dưới đây không chạy mặc định trong notebook để tránh thời gian chạy dài. Chúng phù hợp khi cần đánh giá sensitivity hoặc độ ổn định thống kê hơn:

- **TD($\lambda$) sweep**: thử nhiều giá trị $\lambda$ để xem eligibility trace ảnh hưởng MSE như thế nào.
- **Control epsilon sensitivity**: thử nhiều `epsilon` cho SARSA và Q-learning.
- **Multi-seed smoke**: chạy nhiều random seeds để kiểm tra ổn định nhẹ.

Có thể chạy qua CLI:

```bash
python3 run_experiments.py --verbose 1 --run-td-lambda-sweep
python3 run_experiments.py --verbose 1 --run-control-sensitivity
python3 run_experiments.py --verbose 1 --run-multiseed-smoke
```

Nếu các JSON optional đã tồn tại, cell dưới đây có thể đọc để tham khảo. Đây không phải luồng chính của notebook.


In [ ]:
# Optional/reference only: đọc logs nếu đã chạy CLI sensitivity.
learning_log_dir = PROJECT_ROOT / "logs" / "learning"
td_lambda_sweep = load_json(learning_log_dir / "td_lambda_sweep.json")
control_sensitivity = load_json(learning_log_dir / "control_sensitivity.json")
multiseed_smoke = load_json(learning_log_dir / "multiseed_smoke.json")

if td_lambda_sweep:
    print("TD(lambda) best lambda:", td_lambda_sweep.get("best_lambda_by_mse"))
if control_sensitivity:
    print("SARSA best epsilon:", control_sensitivity.get("sarsa_best_epsilon_by_window_return"))
    print("Q-learning best epsilon:", control_sensitivity.get("q_learning_best_epsilon_by_window_return"))
if multiseed_smoke:
    print("Multi-seed aggregate:", multiseed_smoke.get("aggregate"))

## Tổng kết Learning Algorithms

Learning Algorithms phù hợp khi agent không biết model đầy đủ của môi trường. Thay vì tính Bellman backup từ $P(s' \mid s,a)$, agent học từ sampled transitions sinh bởi `reset()` và `step(action)`.

- TD(0) học state-value function từ one-step TD error.
- TD($\lambda$) dùng eligibility traces để lan truyền TD error về các state trước đó.
- SARSA là on-policy control, học theo policy đang tương tác.
- Q-learning là off-policy control, học theo greedy target.

PolicyEvaluation và ValueIteration đóng vai trò baseline đánh giá, không dùng để training model-free algorithms. Kết quả Learning phụ thuộc mạnh vào hyperparameters, số episodes, exploration strategy và random seed. Vì vậy, các sensitivity experiments và multi-seed checks là bước quan trọng nếu cần kết luận ổn định hơn cho báo cáo.
